In [ ]:
"""Derived portfolios

Shows how to use derived portfolios, a type of portfolio that inherits the contents from a parent portfolio.

Attributes
----------
transactions
holdings
derived portfolios
"""

# Derived portfolios

This notebook demonstrates the LUSID [derived portfolios](https://support.finbourne.com/what-is-a-derived-portfolio). A derived portfolio is a portfolio which inherits the contents (or is "derived") from another parent portfolio. The derived portfolio also contains the entire `transaction` and/or `holding` history of the parent portfolio. You can then modify the data in the derived portfolio without impacting the parent portfolio.

In the example below, we will demonstrate the following workflow:

<ul> (1) Create a parent UK Equity portfolio with some FTSE 100 stocks </ul> 
<ol> (2) Derive a new portfolio from the parent </ol> 
<ol> (3) Cancel a transaction in the derived portfolio but not the parent </ol> 
<ol> (4) Verify that the newly cancelled transaction updates the derived portfolio but not the parent portfolio holdings </ol>

### Setup LUSID

In [ ]:
# Import general purpose packages
import os
import json
from datetime import datetime, timedelta
import pytz

# Import lusid specific packages
import finbourne.sdk.services.lusid as lu
import finbourne.sdk.services.lusid.models as models
from finbourne.sdk.extensions import SyncApiClientFactory, RefreshingToken
from finbourne.sdk.exceptions import ApiException
from finbourne_sdk_utils.pandas_utils.lusid_pandas import lusid_response_to_data_frame
from finbourne_sdk_utils.cocoon.seed_sample_data import seed_data
from finbourne_sdk_utils.cocoon.utilities import create_scope_id

# Import data wrangling packages
import pandas as pd

pd.set_option("display.max_columns", None)

# Authenticate our user and create our API client
secrets_path = os.getenv("FBN_SECRETS_PATH")

# Initiate an API Factory which is the client side object for interacting with LUSID APIs
api_factory = SyncApiClientFactory(
    access_token=RefreshingToken(),
    secrets_path=secrets_path,
    app_name="LusidJupyterNotebook",
)

Load a mapping file for DataFrame headers for the `build transaction` and `get holdings` response.

In [ ]:
with open(r"config/build_transactions_mapping.json") as mappings_file:
    build_transactions_json_mapping = json.load(mappings_file)

with open(r"config/get_holdings_mapping.json") as mappings_file:
    get_holdings_json_mapping = json.load(mappings_file)

### 1) Load default transactions into a new scope

In [ ]:
# Create a new scope

scope = "notebook-derived-portfolios"
portfolio_code = "EQUITY-UK" + "-" + create_scope_id()

In [ ]:
# Load a file of equity transactions

transactions_file = r"data/derived/equity_transactions.csv"
transactions_df = pd.read_csv(transactions_file)
transactions_df["portfolio_code"] = portfolio_code

In [ ]:
# Load portfolios, instruments, and transactions

seed_data_response = seed_data(
    api_factory,
    ["portfolios", "instruments", "transactions"],
    scope,
    transactions_df,
    "DataFrame",
)

In [ ]:
# Define the transaction portfolio API

transaction_portfolio_api = api_factory.build(lu.TransactionPortfoliosApi)
derived_portfolios_api = api_factory.build(lu.DerivedTransactionPortfoliosApi)

### 2) Lets check our holdings

We have:

* 300,000 units in Barclays

In [ ]:
response = transaction_portfolio_api.get_holdings(
    scope=scope, code=portfolio_code, property_keys=["Instrument/default/Name"]
)

holdings_df = lusid_response_to_data_frame(
    response, column_name_mapping=get_holdings_json_mapping, use_camel_case=True
)

holdings_df

### 3) What transactions make up our Barclays holdings?

In [ ]:
build_transactions_response = transaction_portfolio_api.build_transactions(
    scope=scope,
    code=portfolio_code,
    transaction_query_parameters=models.TransactionQueryParameters(
        start_date="2020-01-01", end_date="2020-12-31"
    ),
    property_keys=["Instrument/default/Name"],
)


build_transactions_df = lusid_response_to_data_frame(
    build_transactions_response,
    column_name_mapping=build_transactions_json_mapping,
    use_camel_case=True,
)
build_transactions_df.query("InstrumentName == 'Barclays'")

### 4) Create a derived portfolio

In this section we create a derived portfolio from our parent portfolio. The key message here - all the `transaction` history is inherited from the parent portfolio.

In [ ]:
# Define a scope to hold the derived portfolio

new_scope = "TempReporting" + "-" + scope

print(f"The scope we'll use for the derived portfolios: {new_scope}")

In [ ]:
# Create the derived portfolio

try:
    
    derived_portfolios_api.create_derived_portfolio(scope=new_scope,
                                               create_derived_transaction_portfolio_request = models.CreateDerivedTransactionPortfolioRequest(
                                                   display_name=portfolio_code,
                                                    description="Reporting portfolio",
                                                    code=portfolio_code,
                                                    parent_portfolio_id=models.ResourceId(scope=scope, code=portfolio_code),
                                                    created="2020-01-01T00:00:00Z",
                                                    corporate_action_source_id=None,
                                                    accounting_method=None,
                                                    sub_holding_keys=None,
                                               ))
    
except ApiException as e:
    print(json.loads(e.body)["name"])
    print(json.loads(e.body)["title"])

### 5) Cancel one of the Barclays transactions in the derived portfolio only

In [ ]:
# Cancel one of the transactions with the CancelTransactions endpoint

cancel_response = transaction_portfolio_api.cancel_transactions(
    scope=new_scope, code=portfolio_code, transaction_ids=["trd_0006"]
)

### 6) Check holdings on the primary and the derived portfolio

As expected, we can see:

* The parent portfolio has 300,000 units of Barclays
* The derived portfolio has 150,000 units of Barclays

Run <i>GetHoldings</i> on the parent portfolio

In [ ]:
response = transaction_portfolio_api.get_holdings(
    scope=scope, code=portfolio_code, property_keys=["Instrument/default/Name"]
)

holdings_df = lusid_response_to_data_frame(
    response, column_name_mapping=get_holdings_json_mapping, use_camel_case=True
)

holdings_df.query("InstrumentName == 'Barclays'")

Run <i>GetHoldings</i> on the derived portfolio

In [ ]:
response = transaction_portfolio_api.get_holdings(
    scope=new_scope, code=portfolio_code, property_keys=["Instrument/default/Name"]
)

holdings_df = lusid_response_to_data_frame(
    response, column_name_mapping=get_holdings_json_mapping, use_camel_case=True
)

holdings_df.query("InstrumentName == 'Barclays'")